# <center>**Travaux exploratoires : résumé formaté d'un texte</center>**

# <center>**II. Approche hybride : IA + règles linguistiques</center>**

*Ce fichier est généré sur Jupyter. Pour le faire fonctinner, il faut se placer dans l'environemment coreferee-env.*

*Date de dernière mise à jour : 30/07/2025*

**Méthodologie :** on teste ici une approche **hybride**, mêlant un algorithme d'IA avec l'utilisation de règles linguistiques (simples). SpaCy permet de faire du NER (Reconnaissance d'entités nommées), c'est l'outil idéal pour détecter des personnes, des lieux etc. Il est donc très intéressant pour ces travaux.

- **compréhension du français	:** ✔️ SpaCy est entraîné sur corpus francophones

- **exécution locale sur CPU	:** ✔️ Modèles SpaCy optimisés pour CPU (fr_core_news_md, ~40MB)

- **gratuit et open-source :**	✔️ SpaCy + modèles français sont libres

- **extraction de quadruplets structurés	:** ✔️ Via NER + parsing + règles personnalisées

- **fiabilité :**	✔️ Méthode déterministe, testable, explicable

**Résultats :**

- l'approche hybride est intéressante, toutefois il est difficile et très chronophage de viser une forme d'exhaustivité avec les règles linguistiques
- des problèmes de compatibilité entre les librairies utilisées dans ce notebook (dont SpaCy et numpy) qui n'ont pas pu être résolus


**Améliorations possibles :**
  - régler les problèmes de compatibilité entre librairies Python (utiliser Codex pour trouver les bonnes librairies ?)
  - renforcer les règles linguistiques pour forcer le modèle à mieux détecter les événements, les lieux, les moments et les individus.

**Conclusion :** les difficultés rencontrées (complexité des règles linguistiques et problèmes de compatibilité entre les librairies Python) ont conduit à se tourner vers d'autres approches, purement orientées IA.

### **2.a. Un premier essai**

In [ ]:
import spacy

nlp = spacy.load("fr_core_news_md")

def extract_quadruplets(text):
    doc = nlp(text)
    events = []

    for sent in doc.sents:
        lieu = [ent.text for ent in sent.ents if ent.label_ == "LOC"]
        moment = [ent.text for ent in sent.ents if ent.label_ in ["DATE", "TIME"]]
        individus = [ent.text for ent in sent.ents if ent.label_ == "PER"]
        action = [token.lemma_ for token in sent if token.pos_ == "VERB"]

        if action:
            events.append({
                "événement": ", ".join(action),
                "où": ", ".join(lieu),
                "quand": ", ".join(moment),
                "qui": ", ".join(individus)
            })

    return events

In [ ]:
texte = "Le 21 juin 2023 à 9h15, Jean Dupont est arrivé au 18 rue des Lilas, Paris. Il a assisté à une réunion confidentielle."
quadruplets = extract_quadruplets(texte)

In [ ]:
quadruplets

[{'événement': 'arriver',
  'où': 'rue des Lilas, Paris',
  'quand': '',
  'qui': 'Jean Dupont'},
 {'événement': 'assister', 'où': '', 'quand': '', 'qui': ''}]

**Bilan 2a :**

**Points positifs :**
- le modèle a bien détecté les deux événements
- il les a correctement identifiés
- il a correctement identitfié le lieu lié au premier événement
- il a correctement identifié la personne liée au premier événement

**Points négatifs :**
- n'identifie pas la date du premier événement
- n'identifie pas la personne associée au deuxième événement (toujours Jean Dupont, donc il ne fait pas le lien entre le "Il" de la seconde phrase et "Jean Dupont" de la première phrase
- les descriptions des événements sont trop succints, ils se réduisent à un verbe, sans complément

**Pour info :** ce bilan est transmis à Copilot pour amélioration de la méthode.

### **2.b. Tentative d'amélioration de la méthode**

**Selon Copilot :**  les points négatifs relevés sont classiques quand on utilise SpaCy “brut” : absence de coreference, résumé minimaliste, perte des expressions temporelles non normées…

**Améliorations apportées dans ce qui suit:**

🌍 Détection des lieux (LOC)

📅 Détection des dates et heures (DATE, TIME + regex)

👤 Identification des individus (PER) avec coreference (via coreferee)

🧠 Reconstitution de la phrase verbale complète (verbe + compléments)

🧪 Ajout d’une validation heuristique (exclut les verbes faibles ou abstraits)

**Note :** la gestion de la coréférence a été d'abord testée en essayant d'importer la librairie **coreferee**. Mais cet import s'est avéré impossible après plusieurs heures de tentatives, en raison d'incompatibilités diverses avec des librairies (ou des versions de librairies) utilisées par ailleurs. De nombreux essais (changer les autres librairies, modifier des versions etc.) ont été entreprises mais aucune n'a fonctionné. Il a donc été décidé de ne pas utiliser la libriairie coreferee).

In [ ]:
import spacy
import re

nlp = spacy.load("fr_core_news_md")  # Modèle SpaCy optimisé pour le français

# 💬 Expressions temporelles enrichies
def extract_full_dates(sent):
    date_pattern = r"""(?ix)
        \b(?:le\s*)?
        (?:\d{1,2}(?:er)?\s)?
        (?:janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre)?
        (?:\s\d{4})?
        |\b\d{1,2}/\d{1,2}/\d{4}
        |\b\d{1,2}/\d{1,2}
        |\b\d{4}
    """
    return [m.group().strip() for m in re.finditer(date_pattern, sent.text)]

# 🧠 Phrase verbale complète
def extract_event_phrase(sent):
    for token in sent:
        if token.pos_ == "VERB":
            subtree = sorted(token.subtree, key=lambda x: x.i)
            phrase = " ".join([t.text for t in subtree])
            return phrase
    return ""

# 🔎 Filtre des verbes peu informatifs
def is_valid_event(verb_lemma):
    stop_verbs = {"être", "avoir", "faire", "dire", "sembler", "paraître", "devoir"}
    return verb_lemma not in stop_verbs

# 👥 Mémorisation de l’individu le plus récent
def resolve_pronouns(sent, last_person):
    individu = [ent.text for ent in sent.ents if ent.label_ == "PER"]
    for token in sent:
        if token.text.lower() in {"il", "elle"} and last_person:
            individu.append(last_person)
    return individu

# 🧩 Fonction principale
def extract_quadruplets_enriched(text):
    doc = nlp(text)
    events = []
    last_person = None

    for sent in doc.sents:
        phrase = extract_event_phrase(sent)
        verb_token = next((t for t in sent if t.pos_ == "VERB"), None)
        if not phrase or not verb_token or not is_valid_event(verb_token.lemma_):
            continue

        lieux = [ent.text for ent in sent.ents if ent.label_ == "LOC"]
        moments = [ent.text for ent in sent.ents if ent.label_ in ["DATE", "TIME"]]
        moments += extract_full_dates(sent)

        individu = resolve_pronouns(sent, last_person)

        if individu:
            last_person = individu[-1]  # Mémorise la dernière personne rencontrée

        events.append({
            "événement": phrase,
            "où": ", ".join(lieux),
            "quand": ", ".join(moments),
            "qui": ", ".join(individu)
        })

    return events


**Un exemple pour tester le code précédent**

In [ ]:
texte = """
Compte-rendu d’enquête – Rapport préliminaire

Objet : Enquête sur des activités suspectes signalées dans le quartier des Érables (Secteur 5)

Date : 24 juillet 2025
Responsable de l’enquête : Lieutenant A. Mercier
Durée de l’enquête : Du 18 au 24 juillet 2025

Suite à plusieurs signalements de riverains concernant des allées et venues nocturnes inhabituelles au 17 rue des Marronniers, une enquête de terrain a été ouverte. Les investigations ont débuté par des surveillances discrètes sur une période de cinq nuits consécutives. Les agents ont constaté la présence récurrente de véhicules immatriculés hors département, ainsi que des échanges brefs entre occupants et des individus arrivant à pied.

Une perquisition a été menée le 23 juillet à 6h00, en présence d’un officier de police judiciaire et avec l’accord du parquet. Sur place, les enquêteurs ont découvert plusieurs sachets contenant une substance poudreuse suspectée d’être de la cocaïne (analyse en cours), ainsi qu’un total de 3 200 € en espèces, deux téléphones portables et un carnet mentionnant des noms et montants.

Trois personnes ont été interpellées sur place et placées en garde à vue. L’une d’entre elles est connue des services pour trafic de stupéfiants. Les auditions sont toujours en cours.

L’enquête se poursuit pour établir les filières d’approvisionnement et identifier d’éventuels complices.
"""

In [ ]:
quadruplets = extract_quadruplets_enriched(texte)
for e in quadruplets:
    print(e)

{'événement': 'signalées dans le quartier des Érables ( Secteur 5 ) \n\n', 'où': 'Érables, Secteur 5', 'quand': ', , , , , , , , , , , , , , , , , , , , le, , , , , , , , , , , , , 24 juillet 2025, , , , , , , , , ', 'qui': 'Date, Responsable'}
{'événement': 'concernant des allées et venues nocturnes inhabituelles au 17 rue des Marronniers ,', 'où': 'rue des Marronniers', 'quand': ', , 18, , , 24 juillet 2025, , , , , , , , , , , , , , , , , , , , , , , , , , , , , , 17, , , , , , , , , , , , , , , , , , , , ', 'qui': ''}
{'événement': 'Les investigations ont débuté par des surveillances discrètes sur une période de cinq nuits consécutives .', 'où': '', 'quand': 'Le, , , , , , , , , , , , , , , , , , , , , , , , , , , , , ', 'qui': ''}
{'événement': 'Les agents ont constaté la présence récurrente de véhicules immatriculés hors département , ainsi que des échanges brefs entre occupants et des individus arrivant à pied . \n\n', 'où': '', 'quand': 'Le, , , , , , , , , , , , , , , , , , , 

**Bilan  2b :**

**Points positifs :**
- ok pour la compréhension de à qui il" fait référence dans cet exemple
- ce qui fonctionnait dans la version précédente fonctionne encore

**Points négatifs :**
- plein de virgules inutiles
- apparition de "Mars" non pertinente
- la gestion des références est certainement très peu robuste (exemple : autres pronoms que il ou elle)

### **2c. Gestion des problèmes mentionnés**

**But :**

📅 Dates variées : "le 1er juillet 2023", "juillet 2023", "2023", "01/07/2023", etc.

🔎 Filtrage des doublons et chaînes vides dans les dates

🧹 Nettoyage du champ "quand" pour éviter les valeurs parasites

👤 Remplacement des pronoms par l’entité précédente (type "il" → "Jean Dupont")

In [ ]:
import spacy
import re

nlp = spacy.load("fr_core_news_md")  # modèle français SpaCy

# 🗓️ Expression régulière pour dates variées
def extract_full_dates(sent):
    date_pattern = r"""(?ix)
        \b(?:le\s*)?
        (?:\d{1,2}(?:er)?\s)?
        (?:janvier|février|mars|avril|mai|juin|
           juillet|août|septembre|octobre|novembre|décembre)?
        (?:\s\d{4})?
        |\b\d{1,2}/\d{1,2}/\d{4}
        |\b\d{1,2}/\d{1,2}
        |\b\d{4}
    """

    MONTHS = {
        "janvier", "février", "mars", "avril", "mai", "juin",
        "juillet", "août", "septembre", "octobre", "novembre", "décembre"
    }

    matches = [m.group().strip() for m in re.finditer(date_pattern, sent.text)]
    # 🧹 filtre les mois isolés (ex : "Mars") et les chaînes vides
    results = [r for r in matches if r and r.lower() not in MONTHS and len(r.strip()) > 2]
    return results

# 🔍 Extraction de la phrase verbale
def extract_event_phrase(sent):
    for token in sent:
        if token.pos_ == "VERB":
            phrase = " ".join([t.text for t in sorted(token.subtree, key=lambda x: x.i)])
            return phrase
    return ""

# ⛔️ Verbes trop génériques
def is_valid_event(verb_lemma):
    stop_verbs = {"être", "avoir", "faire", "dire", "sembler", "paraître", "devoir"}
    return verb_lemma not in stop_verbs

# 👥 Logique de co-référence basique
def resolve_pronouns(sent, last_person):
    individu = [ent.text for ent in sent.ents if ent.label_ == "PER"]
    for token in sent:
        if token.text.lower() in {"il", "elle"} and last_person and last_person not in individu:
            individu.append(last_person)
    return individu

# 🧠 Fonction principale
def extract_quadruplets_enriched(text):
    doc = nlp(text)
    events = []
    last_person = None

    for sent in doc.sents:
        phrase = extract_event_phrase(sent)
        verb_token = next((t for t in sent if t.pos_ == "VERB"), None)
        if not phrase or not verb_token or not is_valid_event(verb_token.lemma_):
            continue

        lieux = [ent.text for ent in sent.ents if ent.label_ == "LOC"]

        # 📅 Récupère dates NLP + regex
        moments = [ent.text for ent in sent.ents if ent.label_ in ["DATE", "TIME"]]
        moments += extract_full_dates(sent)
        moments = [m for m in moments if m and len(m.strip()) > 2]
        moments = list(set(moments))  # supprime doublons

        individu = resolve_pronouns(sent, last_person)
        if individu:
            last_person = individu[-1]

        events.append({
            "événement": phrase,
            "où": ", ".join(lieux),
            "quand": ", ".join(moments),
            "qui": ", ".join(individu)
        })

    return events

In [ ]:
quadruplets = extract_quadruplets_enriched(texte)
for e in quadruplets:
    print(e)

{'événement': 'signalées dans le quartier des Érables ( Secteur 5 ) \n\n', 'où': 'Érables, Secteur 5', 'quand': '24 juillet 2025', 'qui': 'Date, Responsable'}
{'événement': 'concernant des allées et venues nocturnes inhabituelles au 17 rue des Marronniers ,', 'où': 'rue des Marronniers', 'quand': '24 juillet 2025', 'qui': ''}
{'événement': 'Les investigations ont débuté par des surveillances discrètes sur une période de cinq nuits consécutives .', 'où': '', 'quand': '', 'qui': ''}
{'événement': 'Les agents ont constaté la présence récurrente de véhicules immatriculés hors département , ainsi que des échanges brefs entre occupants et des individus arrivant à pied . \n\n', 'où': '', 'quand': '', 'qui': ''}
{'événement': 'Une perquisition a été menée le 23 juillet à 6h00 , en présence d’ un officier de police judiciaire et avec l’ accord du parquet .', 'où': '', 'quand': 'le 23 juillet', 'qui': ''}
{'événement': 'Sur place , les enquêteurs ont découvert plusieurs sachets contenant une sub

**Bof**

**Bilan 2c :**

**Points positifs :**

- résolution des principaux problèmes évoqués précédemment

**Points négatifs :**
- "le même jour" n'est pas interprété comme une date
- les deux derniers événements ont été fusionnés dans une même phrase
- les résumés des événements ne sont pas très convaincants : encore des copiers-collers

In [ ]:
# Exemple d'un système d'IA analysant une requête complexe

nlp = spacy.load('fr_core_news_md')

doc = nlp("Quel est le meilleur endroit pour manger des sushis à Paris ?")

for token in doc:

    print(token.text, token.pos_, token.dep_)

Quel ADJ ROOT
est AUX dep
le DET det
meilleur ADJ amod
endroit NOUN nsubj
pour ADP mark
manger VERB advcl
des DET det
sushis NOUN obj
à ADP case
Paris PROPN obl:mod
? PUNCT punct


### **2d. Des tentatives d'amélioration du résumé**

SpaCy est spécialisé dans la tâche de NER (reconnaissance d'entités nommées : lieux, personnes etc.). L'idée maintenant est de le combiner avec un outil de NLP davantage spécialisé dans le résumé de textes (en français).

**Une tentative avec le modèle de résumé local Text_Summarization**

Ce projet open-source propose 3 méthodes extractives (pas génératives) pour résumer un texte en français :

- Mean Summarization : sélection des phrases les plus représentatives via embeddings

- Clustering Summarization : regroupe les phrases par similarité (K-means)

- Graph Summarization : utilise PageRank sur un graphe de similarité entre phrases

📦 Modèles compatibles :

- CamemBERT
- FlauBERT

**On utilise d'abord des règles purement linguistiques, via des expressions régulières**

In [ ]:
import re

def extraire_informations(texte):
    # Définir les motifs pour extraire les informations
    motif_lieu = r"à ([A-ZÉÈÊËÀÂÄÇÎÏÔÖÙÛÜ][a-zéèêëàâäçîïôöùûü]+)"
    motif_annee = r"en (\d{4})"
    motif_personnes = r"\b([A-ZÉÈÊËÀÂÄÇÎÏÔÖÙÛÜ][a-zéèêëàâäçîïôöùûü]+)\b"

    # Trouver les correspondances
    lieux = re.findall(motif_lieu, texte)
    annees = re.findall(motif_annee, texte)
    personnes = re.findall(motif_personnes, texte)

    # Nettoyer les résultats
    lieux = [lieu.strip() for lieu in lieux]
    annees = [annee.strip() for annee in annees]
    personnes = list(set([personne.strip() for personne in personnes if len(personne.strip()) > 2]))

    return lieux, annees, personnes

def generer_resume(texte):
    lieux, annees, personnes = extraire_informations(texte)

    # Générer le résumé
    resume = []
    for i in range(min(len(lieux), len(annees))):
        resume.append({
            "Événement": f"Événement à {lieux[i]}",
            "Où": lieux[i],
            "Quand": annees[i],
            "Qui": personnes[i] if i < len(personnes) else "Inconnu"
        })

    return resume

# Exemple d'utilisation
texte = """
À Paris, en 1987, Claire referma le livre poussiéreux qu'elle venait de découvrir dans le grenier de ses parents. La couverture portait une inscription : "Pour Léon, en souvenir de Lisbonne." Intriguée, elle décida de percer ce mystère.
Trente ans plus tôt, en 1957, Léon traversait les ruelles ensoleillées de Lisbonne, appareil photo en bandoulière. Il photographiait tout : les azulejos, les tramways jaunes, et surtout Elena, la jeune libraire du quartier de l'Alfama, qu'il voyait chaque matin sans jamais oser lui parler.
En 2003, à Montréal, Julien, un étudiant en histoire, tomba sur un cliché ancien exposé dans un café. Au dos, une note : "Elena, Lisbonne, 1957 – L.S." Curieux, il chercha à en savoir plus et retrouva une lettre dans les archives de l'université, signée Claire S.
À Marseille, en 2020, Claire, désormais âgée, raconta à sa petite-fille qu'elle avait retrouvé la trace d'Elena grâce à ce Julien inconnu, qui lui avait envoyé un e-mail accompagné d'une copie du cliché. C'était la première fois qu'elle voyait le visage de celle dont son père avait tant parlé.
Et à Lisbonne, en 2022, Camille, la petite-fille, entra dans la même librairie, désormais tenue par la nièce d'Elena. Le passé semblait vivant entre les étagères.
"""

resume = generer_resume(texte)
for item in resume:
    print(item)


{'Événement': 'Événement à Montréal', 'Où': 'Montréal', 'Quand': '1987', 'Qui': 'Claire'}
{'Événement': 'Événement à Lisbonne', 'Où': 'Lisbonne', 'Quand': '1957', 'Qui': 'Alfama'}


**On souhaite maintenant ajouter une surcouche Spacy**

**Note (26/07) :** ce stade l'import de spacy ne fonctionne pas. Le message d'erreur signale un problème de compatibilité binaire entre numpy et une bibliothèque compilée en C, ici h5py, utilisée indirectement par spaCy via thinc. Après quelques essais infructueux pour installer des (versions) des librairies compatibles entre elles, la gestion des dépendances s'avère plus délicate que prévu. On laisse tomber cette approche pour le moment.

In [ ]:
!python -m spacy download fr_core_news_sm

Traceback (most recent call last):
  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 148, in _get_module_details
  File "<frozen runpy>", line 112, in _get_module_details
  File "C:\Users\olivi\anaconda3\Lib\site-packages\spacy\__init__.py", line 6, in <module>
    from .errors import setup_default_warnings
  File "C:\Users\olivi\anaconda3\Lib\site-packages\spacy\errors.py", line 3, in <module>
    from .compat import Literal
  File "C:\Users\olivi\anaconda3\Lib\site-packages\spacy\compat.py", line 4, in <module>
    from thinc.util import copy_array
  File "C:\Users\olivi\anaconda3\Lib\site-packages\thinc\__init__.py", line 5, in <module>
    from .config import registry
  File "C:\Users\olivi\anaconda3\Lib\site-packages\thinc\config.py", line 5, in <module>
    from .types import Decorator
  File "C:\Users\olivi\anaconda3\Lib\site-packages\thinc\types.py", line 27, in <module>
    from .compat import cupy, has_cupy
  File "C:\Users\olivi\anaconda

**Voici tout de même le code à exécuter pour générer un résumé en utilisant spaCy (uniquement). Si on parvient à importer spaCy et à faire tourner ce code, on pourra ensuite envisager une approche combinée spaCy + utilisation de règles linguistiques par expressions régulières.**

In [ ]:
import spacy

# Charger le modèle français de spaCy
nlp = spacy.load("fr_core_news_sm")

def extraire_entites(texte):
    doc = nlp(texte)

    # Initialiser les listes pour stocker les entités
    lieux = []
    dates = []
    personnes = []

    # Parcourir les entités nommées dans le document
    for ent in doc.ents:
        if ent.label_ == "LOC":  # Lieu
            lieux.append(ent.text)
        elif ent.label_ == "DATE":  # Date
            dates.append(ent.text)
        elif ent.label_ == "PER":  # Personne
            personnes.append(ent.text)

    return lieux, dates, personnes

def generer_resume(texte):
    lieux, dates, personnes = extraire_entites(texte)

    # Générer le résumé
    resume = []
    for i in range(min(len(lieux), len(dates))):
        resume.append({
            "Événement": f"Événement à {lieux[i]}",
            "Où": lieux[i],
            "Quand": dates[i],
            "Qui": personnes[i] if i < len(personnes) else "Inconnu"
        })

    return resume

# Exemple d'utilisation
texte = """
À Paris, en 1987, Claire referma le livre poussiéreux qu'elle venait de découvrir dans le grenier de ses parents. La couverture portait une inscription : "Pour Léon, en souvenir de Lisbonne." Intriguée, elle décida de percer ce mystère.
Trente ans plus tôt, en 1957, Léon traversait les ruelles ensoleillées de Lisbonne, appareil photo en bandoulière. Il photographiait tout : les azulejos, les tramways jaunes, et surtout Elena, la jeune libraire du quartier de l'Alfama, qu'il voyait chaque matin sans jamais oser lui parler.
En 2003, à Montréal, Julien, un étudiant en histoire, tomba sur un cliché ancien exposé dans un café. Au dos, une note : "Elena, Lisbonne, 1957 – L.S." Curieux, il chercha à en savoir plus et retrouva une lettre dans les archives de l'université, signée Claire S.
À Marseille, en 2020, Claire, désormais âgée, raconta à sa petite-fille qu'elle avait retrouvé la trace d'Elena grâce à ce Julien inconnu, qui lui avait envoyé un e-mail accompagné d'une copie du cliché. C'était la première fois qu'elle voyait le visage de celle dont son père avait tant parlé.
Et à Lisbonne, en 2022, Camille, la petite-fille, entra dans la même librairie, désormais tenue par la nièce d'Elena. Le passé semblait vivant entre les étagères.
"""

resume = generer_resume(texte)
for item in resume:
    print(item)



OSError: [E050] Can't find model 'fr_core_news_sm'. It doesn't seem to be a Python package or a valid path to a data directory.